In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('shop.db')

conn.executescript("""
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS customers;

CREATE TABLE customers (
    id INTEGER PRIMARY KEY,
    name TEXT,
    city TEXT,
    signup_date TEXT
);

CREATE TABLE orders (
    id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    product TEXT,
    amount INTEGER,
    order_date TEXT
);

INSERT INTO customers VALUES
(1, 'Анна', 'Москва', '2026-01-15'),
(2, 'Иван', 'Пермь', '2026-02-03'),
(3, 'Мария', 'Екатеринбург', '2026-02-20'),
(4, 'Олег', 'Пермь', '2026-03-11'),
(5, 'Елена', 'Москва', '2026-04-05'),
(6, 'Павел', 'Казань', '2026-05-18');

INSERT INTO orders VALUES
(1, 1, 'Ноутбук', 65000, '2026-02-01'),
(2, 1, 'Мышь', 1500, '2026-02-01'),
(3, 2, 'Монитор', 18000, '2026-03-05'),
(4, 3, 'Клавиатура', 4500, '2026-03-10'),
(5, 2, 'Наушники', 7000, '2026-03-20'),
(6, 4, 'Ноутбук', 58000, '2026-04-02'),
(7, 5, 'Монитор', 21000, '2026-04-15'),
(8, 1, 'Наушники', 9000, '2026-05-01'),
(9, 3, 'Мышь', 1200, '2026-05-07'),
(10, 5, 'Клавиатура', 5200, '2026-05-20'),
(11, 2, 'Ноутбук', 72000, '2026-06-01'),
(12, 4, 'Мышь', 1800, '2026-06-10');
""")
conn.commit()

def q(sql):
    return pd.read_sql(sql, conn)

print('база создана')

база создана


In [2]:
q("SELECT * FROM customers")

,id,name,city,signup_date
0,1,Анна,Москва,2026-01-15
1,2,Иван,Пермь,2026-02-03
2,3,Мария,Екатеринбург,2026-02-20
3,4,Олег,Пермь,2026-03-11
4,5,Елена,Москва,2026-04-05
5,6,Павел,Казань,2026-05-18


In [3]:
q("SELECT name, city FROM customers")

,name,city
0,Анна,Москва
1,Иван,Пермь
2,Мария,Екатеринбург
3,Олег,Пермь
4,Елена,Москва
5,Павел,Казань


In [4]:
q("SELECT * FROM customers WHERE city = 'Пермь'")

,id,name,city,signup_date
0,2,Иван,Пермь,2026-02-03
1,4,Олег,Пермь,2026-03-11


In [5]:
q("SELECT * FROM orders WHERE amount > 10000")

,id,customer_id,product,amount,order_date
0,1,1,Ноутбук,65000,2026-02-01
1,3,2,Монитор,18000,2026-03-05
2,6,4,Ноутбук,58000,2026-04-02
3,7,5,Монитор,21000,2026-04-15
4,11,2,Ноутбук,72000,2026-06-01


In [6]:
q("SELECT * FROM orders WHERE product = 'Мышь' OR amount > 60000")

,id,customer_id,product,amount,order_date
0,1,1,Ноутбук,65000,2026-02-01
1,2,1,Мышь,1500,2026-02-01
2,9,3,Мышь,1200,2026-05-07
3,11,2,Ноутбук,72000,2026-06-01
4,12,4,Мышь,1800,2026-06-10


In [7]:
q("SELECT * FROM orders ORDER BY amount DESC")

,id,customer_id,product,amount,order_date
0,11,2,Ноутбук,72000,2026-06-01
1,1,1,Ноутбук,65000,2026-02-01
2,6,4,Ноутбук,58000,2026-04-02
3,7,5,Монитор,21000,2026-04-15
4,3,2,Монитор,18000,2026-03-05
5,8,1,Наушники,9000,2026-05-01
6,5,2,Наушники,7000,2026-03-20
7,10,5,Клавиатура,5200,2026-05-20
8,4,3,Клавиатура,4500,2026-03-10
9,12,4,Мышь,1800,2026-06-10


In [8]:
q("SELECT product, amount FROM orders ORDER BY amount DESC LIMIT 3")

,product,amount
0,Ноутбук,72000
1,Ноутбук,65000
2,Ноутбук,58000


In [9]:
q("""
SELECT COUNT(*) AS orders_cnt,
       SUM(amount) AS total,
       ROUND(AVG(amount), 2) AS avg_amount,
       MIN(amount) AS min_amount,
       MAX(amount) AS max_amount
FROM orders
""")

,orders_cnt,total,avg_amount,min_amount,max_amount
0,12,264200,22016.67,1200,72000


In [10]:
q("""
SELECT product, COUNT(*) AS cnt, SUM(amount) AS total
FROM orders
GROUP BY product
ORDER BY total DESC
""")

,product,cnt,total
0,Ноутбук,3,195000
1,Монитор,2,39000
2,Наушники,2,16000
3,Клавиатура,2,9700
4,Мышь,3,4500


In [11]:
q("""
SELECT customer_id, COUNT(*) AS orders_cnt, SUM(amount) AS total
FROM orders
GROUP BY customer_id
ORDER BY total DESC
""")

,customer_id,orders_cnt,total
0,2,3,97000
1,1,3,75500
2,4,2,59800
3,5,2,26200
4,3,2,5700


In [12]:
q("""
SELECT c.name, o.product, o.amount
FROM orders o
JOIN customers c ON o.customer_id = c.id
ORDER BY o.amount DESC
LIMIT 5
""")

,name,product,amount
0,Иван,Ноутбук,72000
1,Анна,Ноутбук,65000
2,Олег,Ноутбук,58000
3,Елена,Монитор,21000
4,Иван,Монитор,18000


In [13]:
q("""
SELECT c.name, c.city, SUM(o.amount) AS total
FROM customers c
JOIN orders o ON o.customer_id = c.id
GROUP BY c.id, c.name, c.city
ORDER BY total DESC
""")

,name,city,total
0,Иван,Пермь,97000
1,Анна,Москва,75500
2,Олег,Пермь,59800
3,Елена,Москва,26200
4,Мария,Екатеринбург,5700


In [14]:
q("""
SELECT c.name, COUNT(o.id) AS orders_cnt, COALESCE(SUM(o.amount), 0) AS total
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.id
GROUP BY c.id, c.name
ORDER BY total DESC
""")

,name,orders_cnt,total
0,Иван,3,97000
1,Анна,3,75500
2,Олег,2,59800
3,Елена,2,26200
4,Мария,2,5700
5,Павел,0,0


In [15]:
q("""
SELECT product, SUM(amount) AS total
FROM orders
GROUP BY product
HAVING SUM(amount) > 10000
ORDER BY total DESC
""")

,product,total
0,Ноутбук,195000
1,Монитор,39000
2,Наушники,16000


In [16]:
q("""
SELECT * FROM orders
WHERE amount > (SELECT AVG(amount) FROM orders)
""")

,id,customer_id,product,amount,order_date
0,1,1,Ноутбук,65000,2026-02-01
1,6,4,Ноутбук,58000,2026-04-02
2,11,2,Ноутбук,72000,2026-06-01


In [18]:
q("""
SELECT name FROM customers
WHERE id NOT IN (SELECT customer_id FROM orders)
""")

,name
0,Павел


In [19]:
q("""
WITH totals AS (
    SELECT customer_id, SUM(amount) AS total
    FROM orders
    GROUP BY customer_id
)
SELECT c.name, t.total
FROM totals t
JOIN customers c ON c.id = t.customer_id
WHERE t.total > (SELECT AVG(total) FROM totals)
ORDER BY t.total DESC
""")

,name,total
0,Иван,97000
1,Анна,75500
2,Олег,59800


In [20]:
q("""
SELECT order_date, product, amount,
       SUM(amount) OVER (ORDER BY order_date, id) AS running_total
FROM orders
ORDER BY order_date, id
""")

,order_date,product,amount,running_total
0,2026-02-01,Ноутбук,65000,65000
1,2026-02-01,Мышь,1500,66500
2,2026-03-05,Монитор,18000,84500
3,2026-03-10,Клавиатура,4500,89000
4,2026-03-20,Наушники,7000,96000
5,2026-04-02,Ноутбук,58000,154000
6,2026-04-15,Монитор,21000,175000
7,2026-05-01,Наушники,9000,184000
8,2026-05-07,Мышь,1200,185200
9,2026-05-20,Клавиатура,5200,190400


In [21]:
q("""
SELECT name, product, amount
FROM (
    SELECT c.name, o.product, o.amount,
           ROW_NUMBER() OVER (PARTITION BY c.id ORDER BY o.amount DESC) AS rn
    FROM orders o
    JOIN customers c ON o.customer_id = c.id
) t
WHERE rn = 1
ORDER BY name
""")

,name,product,amount
0,Анна,Ноутбук,65000
1,Елена,Монитор,21000
2,Иван,Ноутбук,72000
3,Мария,Клавиатура,4500
4,Олег,Ноутбук,58000
